In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/team_stats_processed.csv")
future = pd.read_csv("../data/raw/future_games.csv", index_col=0)
print(df.shape, future.shape)
df.head()

(9334, 86) (272, 46)


,season,week,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,...,opp_penalty_yards,opp_timeouts,opp_punt_returns,opp_punt_return_yards,opp_kickoff_returns,opp_kickoff_return_yards,opp_fg_made,opp_fg_att,opp_fg_long,opp_fg_pct
0,2006,1,ARI,SF,23,37,301,3,0,3,...,62,7,3,71,7,169,2,3,44.0,0.666667
1,2006,1,ATL,CAR,10,22,140,2,0,1,...,45,5,3,13,3,64,2,2,54.0,1.000000
2,2006,1,BAL,TB,17,27,181,1,0,1,...,21,4,5,47,6,147,0,0,NaN,NaN
3,2006,1,BUF,NE,15,23,164,0,0,3,...,5,3,1,14,4,82,1,1,32.0,1.000000
4,2006,1,CAR,ATL,21,39,186,0,1,4,...,20,4,3,29,3,87,2,4,32.0,0.500000


In [3]:
# Columns that are game outcomes or unknowable pre-game
META = {"season", "week", "team", "opponent_team", "won",
        "overtime", "is_home", "rest", "opp_rest", "div_game",
        "roof", "surface", "temp", "wind"}

STAT_COLS = [c for c in df.columns if c not in META and not c.startswith("opp_")]

# Sort per-team chronologically, then compute expanding mean of PRIOR games only.
# This fixes the train/predict mismatch: training features now mirror the
# prediction setup (historical averages) instead of the game's own stats.
df_s = df.sort_values(["team", "season", "week"]).reset_index(drop=True)
df_s["won_int"] = df_s["won"].astype(int)

def prior_mean(group):
    out = group[STAT_COLS].shift(1).expanding(min_periods=5).mean()
    out["win_pct"] = group["won_int"].shift(1).expanding(min_periods=5).mean()
    return out

rolled = df_s.groupby("team", group_keys=False).apply(prior_mean)
R_COLS = [f"r_{c}" for c in STAT_COLS] + ["win_pct"]
rolled.columns = R_COLS

META_KEEP = ["season", "week", "team", "opponent_team", "won",
             "is_home", "rest", "opp_rest", "div_game"]
df_roll = pd.concat([df_s[META_KEEP], rolled], axis=1)

# Join each game's opponent rolling stats
opp = df_roll[["season", "week", "team"] + R_COLS].copy()
opp.columns = ["season", "week", "opponent_team"] + [f"opp_{c}" for c in R_COLS]

df_train = df_roll.merge(opp, on=["season", "week", "opponent_team"])
df_train = df_train.dropna(subset=R_COLS + [f"opp_{c}" for c in R_COLS])

# Schedule strength: rolling avg win% of opponents faced so far this season
df_train = df_train.sort_values(["team", "season", "week"]).reset_index(drop=True)
df_train["sos"] = (
    df_train.groupby(["team", "season"])["opp_win_pct"]
    .transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
).fillna(0.5)

# Opponent's SOS from their own perspective (look up from their rows)
sos_lookup = df_train[["season", "week", "team", "sos"]].copy()
sos_lookup.columns = ["season", "week", "opponent_team", "opp_sos"]
df_train = df_train.merge(sos_lookup, on=["season", "week", "opponent_team"], how="left")
df_train["opp_sos"] = df_train["opp_sos"].fillna(0.5)

print(f"{len(df_train)} training rows (min 5 prior games per team required)")

NON_FEATURE = {"season", "week", "team", "opponent_team", "won"}
FEATURE_COLS = [c for c in df_train.columns if c not in NON_FEATURE]
TARGET = "won"
print(f"{len(FEATURE_COLS)} features")

9078 training rows (min 5 prior games per team required)
80 features


In [4]:
imputer = SimpleImputer(strategy="mean")
scaler = MinMaxScaler()

X = pd.DataFrame(
    scaler.fit_transform(imputer.fit_transform(df_train[FEATURE_COLS])),
    columns=FEATURE_COLS,
    index=df_train.index
)
y = df_train[TARGET].astype(int)

X.shape

(9078, 80)

In [5]:
# Tune regularization strength (C = inverse of alpha) via cross-validation
best_C, best_score = 1.0, 0
for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    score = cross_val_score(
        LogisticRegression(C=C, max_iter=1000, random_state=42), X, y,
        cv=TimeSeriesSplit(n_splits=3), scoring="accuracy"
    ).mean()
    print(f"C={C}: {score:.3f}")
    if score > best_score:
        best_score, best_C = score, C

print(f"\nBest C: {best_C}")

C=0.001: 0.533
C=0.01: 0.573
C=0.1: 0.581
C=1.0: 0.575
C=10.0: 0.560

Best C: 0.1


In [6]:
rr = LogisticRegression(C=best_C, max_iter=1000, random_state=42)
split = TimeSeriesSplit(n_splits=3)
sfs = SequentialFeatureSelector(rr, n_features_to_select=30, direction="forward", cv=split, n_jobs=-1)

sfs.fit(X, y)
predictors = list(X.columns[sfs.get_support()])
print(f"Selected {len(predictors)} predictors")
predictors

Selected 30 predictors


['rest',
 'r_completions',
 'r_attempts',
 'r_passing_yards',
 'r_sack_yards_lost',
 'r_passing_first_downs',
 'r_passing_epa',
 'r_rushing_epa',
 'r_def_tackles_for_loss',
 'r_def_sacks',
 'r_def_qb_hits',
 'r_def_fumbles_forced',
 'r_timeouts',
 'r_punt_returns',
 'r_punt_return_yards',
 'r_fg_made',
 'r_fg_long',
 'opp_r_attempts',
 'opp_r_passing_interceptions',
 'opp_r_sacks_suffered',
 'opp_r_passing_epa',
 'opp_r_def_fumbles_forced',
 'opp_r_penalties',
 'opp_r_punt_return_yards',
 'opp_r_kickoff_return_yards',
 'opp_r_fg_made',
 'opp_r_fg_att',
 'opp_r_fg_pct',
 'sos',
 'opp_sos']

In [7]:
df_indexed = df_train.copy()
df_indexed[FEATURE_COLS] = X

seasons = sorted(df_indexed["season"].unique())
results = []

for i, test_season in enumerate(seasons[1:], start=1):
    train = df_indexed[df_indexed["season"] < test_season]
    test  = df_indexed[df_indexed["season"] == test_season]

    rr.fit(train[predictors], train[TARGET].astype(int))
    preds = rr.predict(test[predictors])

    acc = accuracy_score(test[TARGET].astype(int), preds)
    results.append({"test_season": test_season, "train_seasons": i, "accuracy": acc})
    print(f"{test_season}: {acc:.3f}  (trained on {i} season(s))")

results_df = pd.DataFrame(results)
print(f"Mean accuracy: {results_df['accuracy'].mean():.3f}")

2007: 0.593  (trained on 1 season(s))
2008: 0.526  (trained on 2 season(s))
2009: 0.652  (trained on 3 season(s))
2010: 0.554  (trained on 4 season(s))
2011: 0.583  (trained on 5 season(s))
2012: 0.588  (trained on 6 season(s))
2013: 0.562  (trained on 7 season(s))
2014: 0.608  (trained on 8 season(s))
2015: 0.540  (trained on 9 season(s))
2016: 0.639  (trained on 10 season(s))
2017: 0.580  (trained on 11 season(s))
2018: 0.615  (trained on 12 season(s))
2019: 0.571  (trained on 13 season(s))
2020: 0.582  (trained on 14 season(s))
2021: 0.572  (trained on 15 season(s))
2022: 0.552  (trained on 16 season(s))
2023: 0.526  (trained on 17 season(s))
2024: 0.596  (trained on 18 season(s))
2025: 0.610  (trained on 19 season(s))
Mean accuracy: 0.582


In [8]:
# Train final model on all available data
rr.fit(X[predictors], y)
print("Final model trained on all seasons")

Final model trained on all seasons


In [9]:
# Build prediction features using full 2025 season averages (stable team quality estimate)
latest_season = df["season"].max()
latest = df[df["season"] == latest_season].sort_values(["team", "week"]).copy()
latest["won_int"] = latest["won"].astype(int)

form = latest.groupby("team")[STAT_COLS + ["won_int"]].mean().reset_index()
form = form.rename(columns={**{c: f"r_{c}" for c in STAT_COLS}, "won_int": "win_pct"})

# Schedule strength: avg win% of 2025 opponents
team_wpct = latest.groupby("team")["won_int"].mean()
def compute_sos(team_name):
    opps = latest[latest["team"] == team_name]["opponent_team"].values
    return np.mean([team_wpct.get(o, 0.5) for o in opps])

form["sos"] = form["team"].apply(compute_sos)

# Upcoming schedule → long format (one row per team per game)
SCHED_COLS = ["season", "week", "away_team", "home_team",
              "away_rest", "home_rest", "div_game", "roof", "surface", "temp", "wind"]
sched = future[future["game_type"] == "REG"][SCHED_COLS].copy()

home_rows = sched.copy()
home_rows["team"] = home_rows["home_team"]
home_rows["opponent_team"] = home_rows["away_team"]
home_rows["is_home"] = True
home_rows["rest"] = home_rows["home_rest"]
home_rows["opp_rest"] = home_rows["away_rest"]

away_rows = sched.copy()
away_rows["team"] = away_rows["away_team"]
away_rows["opponent_team"] = away_rows["home_team"]
away_rows["is_home"] = False
away_rows["rest"] = away_rows["away_rest"]
away_rows["opp_rest"] = away_rows["home_rest"]

GAME_META = ["season", "week", "team", "opponent_team",
             "is_home", "rest", "opp_rest", "div_game", "roof", "surface", "temp", "wind"]
upcoming = pd.concat([home_rows[GAME_META], away_rows[GAME_META]]).reset_index(drop=True)

opp_form = form.rename(columns={"team": "opponent_team"})
opp_form.columns = ["opponent_team"] + [f"opp_{c}" for c in form.columns if c != "team"]

upcoming = upcoming.merge(form, on="team", how="left")
upcoming = upcoming.merge(opp_form, on="opponent_team", how="left")

X_upcoming = pd.DataFrame(
    scaler.transform(imputer.transform(upcoming[FEATURE_COLS])),
    columns=FEATURE_COLS,
    index=upcoming.index
)

upcoming["predicted_win"] = rr.predict(X_upcoming[predictors])
upcoming[["season", "week", "team", "opponent_team", "is_home", "predicted_win"]]

,season,week,team,opponent_team,is_home,predicted_win
0,2026,1,SEA,NE,True,0
1,2026,1,LA,SF,True,1
2,2026,1,CAR,CHI,True,0
3,2026,1,CIN,TB,True,0
4,2026,1,DET,NO,True,1
...,...,...,...,...,...,...
539,2026,18,CHI,MIN,False,1
540,2026,18,MIA,NE,False,0
541,2026,18,TB,NO,False,1
542,2026,18,PHI,NYG,False,0


In [10]:
# One row per game (home team perspective) with predicted winner
home_view = upcoming[upcoming["is_home"] == True].copy()
home_view["predicted_winner"] = home_view.apply(
    lambda r: r["team"] if r["predicted_win"] == 1 else r["opponent_team"], axis=1
)

home_view[["season", "week", "team", "opponent_team", "predicted_winner"]].sort_values(["week", "team"])

,season,week,team,opponent_team,predicted_winner
2,2026,1,CAR,CHI,CHI
3,2026,1,CIN,TB,TB
4,2026,1,DET,NO,DET
5,2026,1,HOU,BUF,BUF
6,2026,1,IND,BAL,IND
...,...,...,...,...,...
267,2026,18,MIN,CHI,CHI
268,2026,18,NE,MIA,NE
269,2026,18,NO,TB,TB
270,2026,18,NYG,PHI,PHI


In [11]:
matchups = home_view[["season", "week", "opponent_team", "team", "predicted_winner"]].copy()
matchups.columns = ["season", "week", "away_team", "home_team", "predicted_winner"]
matchups = matchups.sort_values(["week", "away_team"]).reset_index(drop=True)

matchups.to_csv("../data/processed/matchup_predictions.csv", index=False)
print(f"Saved {len(matchups)} matchups")
matchups

Saved 272 matchups


,season,week,away_team,home_team,predicted_winner
0,2026,1,ARI,LAC,ARI
1,2026,1,ATL,PIT,PIT
2,2026,1,BAL,IND,IND
3,2026,1,BUF,HOU,BUF
4,2026,1,CHI,CAR,CHI
...,...,...,...,...,...
267,2026,18,PIT,BAL,BAL
268,2026,18,SEA,LA,LA
269,2026,18,SF,ARI,SF
270,2026,18,TB,NO,TB


In [12]:
upcoming.to_csv("../data/processed/predictions.csv", index=False)
print(f"Saved {len(upcoming)} rows")

Saved 544 rows
